# Isolation Forest

* Designed specifically for anomaly detection
* Anomalies are few and different, and can therefore be isolated with fewer random splits than normal points.
* Isolation Forest does not rely on distance or density estimation. It isolates points using random partitions.
* Imagine you want to isolate a point by repeatedly splitting the feature space using random rules (like choosing a random dimension, then a random threshold).
* Normal points exist in dense clusters → they require many random splits to isolate.
* Anomalies exist in sparse regions → they require fewer splits to isolate.

# Steps

To build one tree:

* Randomly sample $\phi$ points from the dataset (ψ is the sub-sample size, often 256).
* If the node has 1 point, or the depth has reached max height then stop splitting.
* Else, Randomly select a feature $x_j$ and randomly choose a split value between its min and max.
* Partition the data into left and right subsets.
* Recursively repeat.
* This grows a random binary tree
* Repeat the tree-building process t times.
* For each point, go through every tree and measure path length (number of edges from root to the leaf)

# Mathematical Formulation

If the traversal ends in a pure leaf (only 1 sample):
$$
\LARGE h(x)=\text{depth}(x)
$$

If it ends in a leaf with multiple samples z (e.g., z = 5):

$$
\LARGE h(x)=\text{depth}(x)+c(z)
$$

$c(z)$ is the expected extra path length needed to isolate the remaining points.
$$
\LARGE c(z) = 2 H_{z-1} - \frac{2(z-1)}{z}
$$

$$
\LARGE H_n = \sum_{i=1}^{n} \frac{1}{i}
$$


**Normalized path length for a sample is given by**
$$
\LARGE h(x) = \frac{1}{t} \sum_{i=1}^{t} h_i(x)
$$

**Final Anomalie Score for a point is given by**
$$
\LARGE s(x) = 2^{-\frac{h(x)}{c(\psi)}}
$$

* If s(x)≈1, the point is highly anomalous
* If s(x)≈0.5, the point is normal
* If s(x)<0.5, the point is very normal

# Why subsampling is required ? 

In a small random subsample:
* Ensures anomalies are rare in each sample
* Normal points are fewer
* Anomalies usually appear alone
* A few random splits isolate them quickly
* So anomalies get short path lengths, which is the core idea.

# Implementation

In [11]:
class ITreeNode:
    def __init__(self, depth=0, size=0, feature=None, threshold=None):
        self.depth = depth
        self.size = size  # Number of points in leaf
        self.feature = feature
        self.threshold = threshold
        self.left = None
        self.right = None


In [12]:
import numpy as np

class IsolationTree:
    def __init__(self, max_depth):
        self.max_depth = max_depth
        self.root = None

    def fit(self, X):
        self.root = self._build_tree(X, current_depth=0)

    def _build_tree(self, X, current_depth):
        n_samples, n_features = X.shape

        # Stop conditions
        if current_depth >= self.max_depth or n_samples <= 1:
            leaf = ITreeNode(depth=current_depth, size=n_samples)
            return leaf

        # Randomly select a feature
        feature = np.random.randint(0, n_features)
        min_val = X[:, feature].min()
        max_val = X[:, feature].max()

        if min_val == max_val:  # No further split possible
            leaf = ITreeNode(depth=current_depth, size=n_samples)
            return leaf

        # Randomly choose a split threshold
        threshold = np.random.uniform(min_val, max_val)

        # Partition the data
        left_mask = X[:, feature] < threshold
        right_mask = X[:, feature] >= threshold

        node = ITreeNode(depth=current_depth, feature=feature, threshold=threshold)
        node.left = self._build_tree(X[left_mask], current_depth + 1)
        node.right = self._build_tree(X[right_mask], current_depth + 1)

        return node

    # Compute path length for a single point
    def path_length(self, x, node=None):
        if node is None:
            node = self.root

        # Leaf
        if node.left is None and node.right is None:
            # c(n) adjustment for leaf with multiple points
            if node.size <= 1:
                return node.depth
            else:
                return node.depth + 2 * (np.log(node.size - 1 + 1e-10) + 0.5772156649) - 2*(node.size-1)/node.size

        # Recurse
        if x[node.feature] < node.threshold:
            return self.path_length(x, node.left)
        else:
            return self.path_length(x, node.right)


In [13]:
class IsolationForest:
    def __init__(self, n_trees=100, sample_size=256):
        self.n_trees = n_trees
        self.sample_size = sample_size
        self.trees = []

    def fit(self, X):
        self.trees = []
        max_depth = int(np.ceil(np.log2(self.sample_size)))
        n_samples = X.shape[0]

        for _ in range(self.n_trees):
            # Subsample without replacement
            if n_samples <= self.sample_size:
                X_sample = X
            else:
                idx = np.random.choice(n_samples, self.sample_size, replace=False)
                X_sample = X[idx]

            tree = IsolationTree(max_depth)
            tree.fit(X_sample)
            self.trees.append(tree)

    # Compute anomaly score for all points
    def anomaly_score(self, X):
        scores = []
        c_psi = 2 * (np.log(self.sample_size - 1) + 0.5772156649) - 2*(self.sample_size-1)/self.sample_size

        for x in X:
            path_lengths = np.array([tree.path_length(x) for tree in self.trees])
            h = path_lengths.mean()
            score = 2 ** (-h / c_psi)
            scores.append(score)
        return np.array(scores)


In [16]:
# Small numerical dataset
X = np.array([
    [1.0, 2.0],
    [1.1, 2.1],
    [0.9, 1.8],
    [10.0, 10.0],   # anomaly
    [1.2, 2.2],
    [0.8, 1.9]
])

# Fit Isolation Forest
iso = IsolationForest(n_trees=200, sample_size=4)
iso.fit(X)

# Compute anomaly scores
scores = iso.anomaly_score(X)
for i, score in enumerate(scores):
    print(f"Point {X[i]} -> Anomaly Score: {score:.3f}")


Point [1. 2.] -> Anomaly Score: 0.455
Point [1.1 2.1] -> Anomaly Score: 0.467
Point [0.9 1.8] -> Anomaly Score: 0.478
Point [10. 10.] -> Anomaly Score: 0.617
Point [1.2 2.2] -> Anomaly Score: 0.486
Point [0.8 1.9] -> Anomaly Score: 0.477
